In [9]:
from google.colab import files
uploaded = files.upload()

Saving features_2000.csv to features_2000 (1).csv


In [10]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

df = pd.read_csv('features_2000.csv')
print(len(df), 'raw')

df = df.drop_duplicates(subset='kepoi_name', keep='last')
print(len(df), 'after dedup')      # expect 2289

ok = df[df['error'].isna()].copy()
print(len(ok), 'successful rows')

FEATURES = ['depth', 'width', 'asymmetry', 'snr',
            'odd_even_diff', 'secondary_depth', 'duration_over_period']

X = ok[FEATURES]
y = (ok['disposition'] == 'CONFIRMED').astype(int)
groups = ok['kepid']

2295 raw
2289 after dedup
2224 successful rows


In [11]:
outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(outer.split(X, y, groups))

inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
a, b = next(inner.split(X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]))
tr, va = train_idx[a], train_idx[b]

In [12]:
# NOTE: untuned, matching the final model below.
# Grid search (next cell) gave CV 0.8806 vs default 0.8754 — not adopted.

m = HistGradientBoostingClassifier(random_state=42).fit(X.iloc[tr], y.iloc[tr])
pv = m.predict_proba(X.iloc[va])[:, 1]

for t in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    p = (pv >= t).astype(int)
    tp = ((p == 1) & (y.iloc[va] == 1)).sum()
    fp = ((p == 1) & (y.iloc[va] == 0)).sum()
    fn = ((p == 0) & (y.iloc[va] == 1)).sum()
    print(f"{t}: recall {tp/(tp+fn):.2f}, precision {tp/(tp+fp):.2f}")

0.1: recall 0.97, precision 0.72
0.15: recall 0.96, precision 0.74
0.2: recall 0.96, precision 0.76
0.25: recall 0.96, precision 0.77
0.3: recall 0.94, precision 0.78
0.4: recall 0.93, precision 0.78
0.5: recall 0.90, precision 0.79


In [ ]:
# from sklearn.model_selection import GridSearchCV, GroupKFold

# grid = {
#     'learning_rate': [0.05, 0.1],
#     'max_leaf_nodes': [15, 31],
#     'min_samples_leaf': [20, 40],
#     'l2_regularization': [0.0, 1.0],
#     'max_iter': [100, 300],
# }

# search = GridSearchCV(
#     HistGradientBoostingClassifier(random_state=42),
#     grid,
#     scoring='average_precision',
#     cv=GroupKFold(n_splits=5),
#     n_jobs=-1,
# )

# search.fit(X.iloc[train_idx], y.iloc[train_idx], groups=groups.iloc[train_idx])

# print(search.best_params_)
# print(f"best CV score: {search.best_score_:.4f}")

# import numpy as np
# default_score = search.cv_results_['mean_test_score'][
#     np.where((search.cv_results_['param_learning_rate'] == 0.1) &
#              (search.cv_results_['param_max_leaf_nodes'] == 31) &
#              (search.cv_results_['param_min_samples_leaf'] == 20) &
#              (search.cv_results_['param_l2_regularization'] == 0.0) &
#              (search.cv_results_['param_max_iter'] == 100))[0][0]
# ]
# print(f"default score: {default_score:.4f}")

In [13]:
THRESHOLD = 0.25   # <- from cell 4

final = HistGradientBoostingClassifier(random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
probs = final.predict_proba(X.iloc[test_idx])[:, 1]
preds = (probs >= THRESHOLD).astype(int)

print(classification_report(y.iloc[test_idx], preds,
                            target_names=['FALSE POSITIVE', 'CONFIRMED']))
print(confusion_matrix(y.iloc[test_idx], preds))

imp = permutation_importance(final, X.iloc[test_idx], y.iloc[test_idx],
                             n_repeats=20, random_state=42)
for n, mn, sd in sorted(zip(FEATURES, imp.importances_mean, imp.importances_std),
                        key=lambda t: -t[1]):
    print(f"{n:22s} {mn:+.4f} ± {sd:.4f}")

                precision    recall  f1-score   support

FALSE POSITIVE       0.94      0.80      0.86       301
     CONFIRMED       0.66      0.88      0.75       133

      accuracy                           0.82       434
     macro avg       0.80      0.84      0.81       434
  weighted avg       0.85      0.82      0.83       434

[[240  61]
 [ 16 117]]
duration_over_period   +0.1248 ± 0.0138
secondary_depth        +0.0620 ± 0.0099
snr                    +0.0446 ± 0.0110
depth                  +0.0225 ± 0.0099
width                  +0.0185 ± 0.0060
odd_even_diff          +0.0158 ± 0.0118
asymmetry              +0.0066 ± 0.0090


In [ ]:
# print(y.iloc[va].mean(), 'CONFIRMED rate in validation')
# print(y.iloc[test_idx].mean(), 'CONFIRMED rate in test')

In [14]:
import joblib

joblib.dump({
    'model': final,
    'features': FEATURES,
    'threshold': THRESHOLD,
}, 'transitcheck_model.joblib')

from google.colab import files
files.download('transitcheck_model.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>